# Python: Functional Programming

---

## Course Overview

This notebook covers the core principles of functional programming applied in Python:

| # | Module | Core Concept |
|---|--------|--------------|
| 1 | Introduction to Functional Programming | Paradigms & History |
| 2 | First-Class Functions | Higher-order functions, closures, currying |
| 3 | Pure Functions | Side-effect elimination, lambdas |
| 4 | Immutable Variables | Immutability rules, frozen structures |
| 5 | Lazy Evaluation | Generators, iterators, eager vs. lazy |
| 6 | Recursion | Tail recursion, trampolining |
| 7 | Pattern Matching | `match`/`case`, guards, structural patterns |

---

## Module 1 — Introduction to Functional Programming

### Programming Paradigms

Programming paradigms represent fundamentally different approaches to expressing computation:

| Paradigm | Examples | Approach |
|----------|----------|----------|
| **Imperative** | Machine code, Assembly | Sequence of instructions that change program state |
| **Procedural** | Fortran, COBOL, C | Grouped instructions into reusable procedures |
| **Object-Oriented** | Java, C++, Python | State and behaviour bundled into objects |
| **Declarative** | SQL, HTML | Describe *what* you want, not *how* |
| **Logic** | Prolog, Planner | Express facts and rules; runtime infers solutions |
| **Functional** | Haskell, ML, Lisp | Computation as evaluation of mathematical functions |

> Python supports **multiple paradigms**, making it a great language for learning FP without abandoning OOP.

### A Brief History of Functional Programming

```
1950s  →  1960s  →  1970s  →  1970s  →  1980s
Lisp      APL       FP        ML        Haskell
(McCarthy)(Iverson) (Backus)  (Milner)  (committee)
```

### Core Functional Programming Principles

Functional programming is built on six key principles:

1. **First-class functions** — functions are values like any other object
2. **Pure functions** — same input always produces the same output, no side effects
3. **Immutable variables and objects** — data does not change after creation
4. **Lazy evaluation** — delay computation until results are actually needed
5. **Recursion** — express loops as recursive function calls
6. **Pattern matching** — declarative branching on data structure and value

> **Key insight:** FP does not exclude OOP. Python lets you mix both paradigms strategically.

In [1]:
# The functional mindset — think in transformations, not mutations

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Imperative style
evens_imperative = []
for n in numbers:
    if n % 2 == 0:
        evens_imperative.append(n)

# Functional style — describe WHAT, not HOW
evens_functional = list(filter(lambda n: n % 2 == 0, numbers))
evens_comprehension = [n for n in numbers if n % 2 == 0]

print('Imperative:', evens_imperative)
print('Functional:', evens_functional)
print('Comprehension:', evens_comprehension)

Imperative: [2, 4, 6, 8, 10]
Functional: [2, 4, 6, 8, 10]
Comprehension: [2, 4, 6, 8, 10]


---

## Module 2 — Reaching Higher with First-Class Functions

### What Are First-Class Functions?

A language has **first-class functions** when functions are treated as values:
- Can be **assigned** to variables
- Can be **passed as arguments** to other functions
- Can be **returned** from other functions
- Can be **stored** in data structures

Python supports all of these — functions are objects.

### Higher-Order Functions

A **higher-order function** either:
- Accepts a function as an argument, or
- Returns a function

Three key patterns:

In [2]:
# ── Pattern 1: Composition ────────────────────────────────────────────────────
# Pass a function as an argument to another function

def f(x):
    return x + 2

def g(h, x):
    return h(x) * 2

print('Composition:', g(f, 42))  # f(42)=44, then 44*2=88

Composition: 88


In [3]:
# ── Pattern 2: Closure ────────────────────────────────────────────────────────
# A function that captures variables from its enclosing scope

def addx(x):
    def _(y):
        return x + y   # 'x' is captured from the outer scope
    return _

add2 = addx(2)
add3 = addx(3)

print('add2(5):', add2(5))   # 7
print('add3(5):', add3(5))   # 8
print('add2(2), add3(3):', add2(2), add3(3))   # 4, 6

add2(5): 7
add3(5): 8
add2(2), add3(3): 4 6


In [4]:
# ── Pattern 3: Currying ───────────────────────────────────────────────────────
# Transform a function with N arguments into N functions with 1 argument each
# n-arity → n-1-arity → ... → 1-arity

def f(x, y):
    return x * y

def f2(x):
    def _(y):
        return f(x, y)
    return _

print('f2(2) is a function:', f2(2))   # function object
print('f2(2)(3):', f2(2)(3))           # 6

# 3-argument currying example
def g(x, y, z):
    return x * y - z

def g1(x, y):
    return lambda z: g(x, y, z)

def g2(x):
    return lambda y: g1(x, y)

print('g(1,2,3):', g(1, 2, 3))         # 1*2-3 = -1
print('g1(1,2)(3):', g1(1, 2)(3))      # same
print('g2(1)(2)(3):', g2(1)(2)(3))     # fully curried

f2(2) is a function: <function f2.<locals>._ at 0x10b0bd6f0>
f2(2)(3): 6
g(1,2,3): -1
g1(1,2)(3): -1
g2(1)(2)(3): -1


In [5]:
# ── functools.partial: partial application (related to currying) ──────────────
# Pre-fill some arguments of a function, creating a new function with fewer args

from functools import partial

def multiply(x, y):
    return x * y

double = partial(multiply, 2)   # pre-fill x=2
triple = partial(multiply, 3)   # pre-fill x=3

print('double(5):', double(5))   # 10
print('triple(5):', triple(5))   # 15

nums = [1, 2, 3, 4, 5]
print('doubled list:', list(map(double, nums)))
print('tripled list:', list(map(triple, nums)))

double(5): 10
triple(5): 15
doubled list: [2, 4, 6, 8, 10]
tripled list: [3, 6, 9, 12, 15]


In [6]:
# ── Built-in higher-order functions ──────────────────────────────────────────
# map, filter, sorted — foundational tools for functional Python

data = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]

# map: apply a function to every item
squared = list(map(lambda x: x**2, data))
print('Squared:', squared)

# filter: keep items where predicate is True
evens = list(filter(lambda x: x % 2 == 0, data))
print('Evens:', evens)

# sorted with a key function
words = ['banana', 'apple', 'cherry', 'date']
by_length = sorted(words, key=len)
print('By length:', by_length)

# Combine map + filter
even_squares = list(map(lambda x: x**2, filter(lambda x: x % 2 == 0, data)))
print('Even squares:', even_squares)

Squared: [9, 1, 16, 1, 25, 81, 4, 36, 25, 9, 25]
Evens: [4, 2, 6]
By length: ['date', 'apple', 'banana', 'cherry']
Even squares: [16, 4, 36]


---

## Module 3 — Eliminating Side Effects with Pure Functions

### What Is a Pure Function?

A **pure function** satisfies all of these properties:

| Property | Description |
|----------|-------------|
| **Single responsibility** | Does one thing and does it well |
| **Limited arguments** | Minimal parameters |
| **Output depends on input** | No hidden state, no reading from external sources |
| **Referential transparency** | Same input → same output, every time |
| **No state mutation** | Does not read or modify external state |
| **No side effects** | No I/O, no mutation of arguments |
| **Composable** | Build complexity by combining small pure functions |

### Side Effects

A **side effect** is any observable action beyond returning a value:
- Modifying a global or mutable argument
- Writing to a file or database
- Printing to stdout
- Raising exceptions as control flow
- Reading from external state (time, random numbers, etc.)

> **Goal:** Push side effects to the edges of your program. Keep the core logic pure.

In [7]:
# ── Impure vs. Pure: mutation example ────────────────────────────────────────

# IMPURE — modifies a shared list
shared_list = [1, 2, 3]

def impure_append(lst, item):
    lst.append(item)    # side effect: mutates the caller's list
    return lst

impure_append(shared_list, 4)
print('After impure_append:', shared_list)  # original is changed!


# PURE — returns a new list, does not touch the original
original = [1, 2, 3]

def pure_append(lst, item):
    return lst + [item]   # creates a new list

new_list = pure_append(original, 4)
print('original:', original)   # unchanged
print('new_list:', new_list)

After impure_append: [1, 2, 3, 4]
original: [1, 2, 3]
new_list: [1, 2, 3, 4]


In [8]:
# ── One function vs. three: SRP in action ─────────────────────────────────────
# From the slides: the 'do it all' function is harder to reason about

# BAD — one function doing three things
def get_ints_bad(ints, odd=True, even=True):
    if odd and even:
        return [i for i in ints]
    elif odd:
        return [i for i in ints if i % 2]
    elif even:
        return [i for i in ints if not i % 2]
    else:
        return []

# GOOD — three focused pure functions
def get_even_ints(ints):
    return [i for i in ints if not i % 2]

def get_odd_ints(ints):
    return [i for i in ints if i % 2]

def get_all_ints(ints):
    return list(ints)

nums = range(10)
print('All:  ', get_all_ints(nums))
print('Even: ', get_even_ints(nums))
print('Odd:  ', get_odd_ints(nums))

All:   [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Even:  [0, 2, 4, 6, 8]
Odd:   [1, 3, 5, 7, 9]


In [9]:
# ── Lambda functions ──────────────────────────────────────────────────────────
# lambdas are anonymous single-expression functions
# Rule of thumb: prefer named helper functions when used more than once

# Lambda: useful inline, one-off
square = lambda x: x ** 2
print('square(7):', square(7))

# Named helper: prefer when reusability or clarity matters
def square_named(x):
    return x ** 2

data = [1, 2, 3, 4, 5]
print('lambda map:  ', list(map(lambda x: x**2, data)))
print('helper map:  ', list(map(square_named, data)))

square(7): 49
lambda map:   [1, 4, 9, 16, 25]
helper map:   [1, 4, 9, 16, 25]


---

## Module 4 — Reducing Data Errors with Immutable Variables

### Immutability Rules

Functional programming imposes two rules on variables:

1. **Local variables do not change** — once assigned inside a function, they stay the same
2. **Global variables can only change their reference** — never mutate the object in place

Why mutable state is dangerous:
- Hidden dependencies between functions
- Race conditions in concurrent code
- Surprising bugs from shared state

### Python's Immutable Built-ins

| Mutable | Immutable equivalent |
|---------|---------------------|
| `list` | `tuple` |
| `dict` | `types.MappingProxyType` |
| `set` | `frozenset` |
| Regular class | `@dataclass(frozen=True)` / `namedtuple` |

In [10]:
# ── Demonstrating variable scoping and mutation ───────────────────────────────

pi = 3.14159

# This does NOT change the global pi — it creates a local shadow
def try_change_pi(pi):
    pi = 2.71828   # local rebind, no effect on global
    return pi

# This DOES change the global pi — avoid this pattern!
def really_change_pi(*args, **kwargs):
    global pi
    pi = 2.71828

print('Before:', pi)
try_change_pi(pi)
print('After try_change_pi:', pi)  # still 3.14159

really_change_pi()
print('After really_change_pi:', pi)  # now 2.71828 — dangerous!

Before: 3.14159
After try_change_pi: 3.14159
After really_change_pi: 2.71828


In [11]:
# ── Immutable data structures ─────────────────────────────────────────────────

# tuple — immutable list
point = (3, 4)
try:
    point[0] = 10
except TypeError as e:
    print('tuple:', e)

# frozenset — immutable set
fs = frozenset([1, 2, 3])
try:
    fs.add(4)
except AttributeError as e:
    print('frozenset:', e)

# Frozen dataclass — immutable objects
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(3.0, 4.0)
print('Point:', p)

try:
    p.x = 10.0
except Exception as e:
    print('frozen dataclass:', type(e).__name__, e)

tuple: 'tuple' object does not support item assignment
frozenset: 'frozenset' object has no attribute 'add'
Point: Point(x=3.0, y=4.0)
frozen dataclass: FrozenInstanceError cannot assign to field 'x'


In [12]:
# ── Custom Immutable base class (from the course demos) ───────────────────────
import abc

class Immutable(abc.ABC):
    """ABC that prevents attribute modification after initial assignment."""
    __slots__ = ('__attrs__',)

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.__attrs__ = frozenset()

    def __setattr__(self, name, value):
        if name == '__attrs__':
            super().__setattr__(name, value)
            return
        if name in self.__attrs__:
            raise AttributeError(f"Attempt to modify immutable attribute '{name}'")
        else:
            super().__setattr__(name, value)
            self.__attrs__ |= {name}

    def __delattr__(self, name):
        if name in self.__attrs__:
            raise AttributeError(f"Attempt to delete immutable attribute '{name}'")
        raise AttributeError(name)


class Customer(Immutable):
    def __init__(self, name, email):
        super().__init__()
        self.name = name
        self.email = email

c = Customer('Alice', 'alice@example.com')
print('Customer:', c.name, c.email)

try:
    c.name = 'Bob'
except AttributeError as e:
    print('Immutable protection:', e)

Customer: Alice alice@example.com
Immutable protection: Attempt to modify immutable attribute 'name'


In [13]:
# ── Functional update pattern: produce new objects instead of mutating ─────────
# Use dataclasses.replace() or manual construction

from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Order:
    order_id: int
    status: str
    amount: float

order = Order(order_id=1, status='pending', amount=99.99)
print('Original:', order)

# Instead of mutating, create a new instance with updated fields
shipped_order = replace(order, status='shipped')
print('Updated:', shipped_order)
print('Original unchanged:', order)

Original: Order(order_id=1, status='pending', amount=99.99)
Updated: Order(order_id=1, status='shipped', amount=99.99)
Original unchanged: Order(order_id=1, status='pending', amount=99.99)


---

## Module 5 — Avoiding Unnecessary Work Using Lazy Evaluation

### Eager vs. Lazy Evaluation

| | **Eager** | **Lazy** |
|---|-----------|----------|
| When evaluated | When the statement is reached | When the result is actually needed |
| Assignment | Computes immediately | Deferred |
| Cost model | Pay up front | Pay as you go |
| Best when | Most results are used | Only a subset of results are needed |
| Risk | Wasteful if some results go unused | Recomputes if iterated multiple times |

### Lazy Evaluation in Python — Timeline

| Version | Feature |
|---------|--------|
| 1.4 | `xrange()` |
| 1.5 | `__getitem__()` protocol |
| 2.2 | Iterators, generators, `yield` |
| 3.0 | `map` and `filter` return lazy iterators |
| 3.3 | `yield from` |

> Python 3 made **lazy evaluation the default** for many built-ins.

In [14]:
# ── Eager vs. Lazy: map and list ──────────────────────────────────────────────
import sys

data = range(1_000_000)

# Eager — builds the whole list in memory immediately
eager = [x * 2 for x in data]
print(f'Eager list size:  {sys.getsizeof(eager):,} bytes')

# Lazy — map object; computes each element on demand
lazy = map(lambda x: x * 2, data)
print(f'Lazy map size:    {sys.getsizeof(lazy):,} bytes')
print(f'Type: {type(lazy)}')

Eager list size:  8,448,728 bytes
Lazy map size:    56 bytes
Type: <class 'map'>


In [15]:
# ── Generator functions with yield ────────────────────────────────────────────
# Generators produce values one at a time — memory-efficient for large sequences

def infinite_counter(start=0):
    n = start
    while True:
        yield n
        n += 1

counter = infinite_counter()
first_five = [next(counter) for _ in range(5)]
print('First five:', first_five)

# Generator expression — like a list comprehension, but lazy
gen_expr = (x**2 for x in range(10))
print('Generator expression type:', type(gen_expr))
print('Values:', list(gen_expr))

First five: [0, 1, 2, 3, 4]
Generator expression type: <class 'generator'>
Values: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [16]:
# ── itertools: lazy combinatorial tools ──────────────────────────────────────
import itertools

# islice: take a slice without materialising the whole sequence
squares = (x**2 for x in itertools.count(1))
first_10_squares = list(itertools.islice(squares, 10))
print('First 10 squares:', first_10_squares)

# chain: concatenate iterables lazily
chained = itertools.chain([1, 2, 3], [4, 5], [6])
print('Chained:', list(chained))

# takewhile: yield while predicate is True
data = itertools.count(1)
under_10 = list(itertools.takewhile(lambda x: x < 10, data))
print('takewhile < 10:', under_10)

First 10 squares: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Chained: [1, 2, 3, 4, 5, 6]
takewhile < 10: [1, 2, 3, 4, 5, 6, 7, 8, 9]


In [17]:
# ── yield from: delegate to a sub-generator ──────────────────────────────────

def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # delegate recursively
        else:
            yield item

nested_list = [1, [2, 3], [4, [5, 6]], 7]
print('Flattened:', list(flatten(nested_list)))

Flattened: [1, 2, 3, 4, 5, 6, 7]


In [18]:
# ── Tips for using generators ─────────────────────────────────────────────────
# From the slides: Using Generators
#   - If processing is expensive, materialise once with tuple()
#   - Not directly indexable → use itertools.islice or itertools.tee
#   - Profile before optimising

def expensive_compute(n):
    return n ** 3  # imagine this is slow

# Compute once, iterate many times
computed = tuple(expensive_compute(x) for x in range(5))
print('Materialised tuple:', computed)
print('Can index:', computed[2])
print('Can iterate again:', list(computed))

Materialised tuple: (0, 1, 8, 27, 64)
Can index: 8
Can iterate again: [0, 1, 8, 27, 64]


---

## Module 6 — Replacing Iteration with Recursion

### Foundations of Recursion

Recursion has a **mathematical foundation** in recurrence relations:

**Fibonacci:**
$$F_n = \begin{cases} 0 & \text{if } n = 0 \\ 1 & \text{if } n = 1 \\ F_{n-1} + F_{n-2} & \text{if } n > 1 \end{cases}$$

**Sum:**
$$S_n = \begin{cases} 0 & \text{if } n = 0 \\ n + S_{n-1} & \text{if } n > 0 \end{cases}$$

E.W. Dijkstra argued for structured programming (no GOTOs) — recursion is the structured alternative to explicit loops.

### Tail Recursion

A function is **tail recursive** when the recursive call is the **last operation** — there is nothing left to do after it returns.

Benefits:
- Only one stack frame needed at a time
- Can be mechanically converted to a loop

**Python caveat:** Python does **not** optimise tail calls. The stack limit is ~1000 by default. The workaround is **trampolining**.

In [19]:
# ── Basic recursion: Fibonacci ────────────────────────────────────────────────

def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print('Fibonacci 0-9:', [fibonacci(i) for i in range(10)])

Fibonacci 0-9: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


In [20]:
# ── Tail-recursive Fibonacci ──────────────────────────────────────────────────
# Accumulator pattern: carry the partial result in an extra argument

def fibonacci_tail(n, curr=0, next_val=1):
    if n == 0:
        return curr
    return fibonacci_tail(n - 1, next_val, curr + next_val)

print('Tail Fibonacci 0-9:', [fibonacci_tail(i) for i in range(10)])

# Same pattern for sum
def rsum(lst):
    if not lst:
        return 0
    return lst[0] + rsum(lst[1:])

def rsum_tail(lst, acc=0):
    if not lst:
        return acc
    return rsum_tail(lst[1:], acc + lst[0])

print('rsum([1..5]):', rsum([1, 2, 3, 4, 5]))
print('rsum_tail([1..5]):', rsum_tail([1, 2, 3, 4, 5]))

Tail Fibonacci 0-9: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
rsum([1..5]): 15
rsum_tail([1..5]): 15


In [21]:
# ── Trampolining: workaround for Python's stack limit ─────────────────────────
# Because Python does not eliminate tail calls, we simulate it with generators.
# A 'trampoline' runs the generator step-by-step without growing the call stack.

import types

def tramp(gen, *args, **kwargs):
    """
    Trampoline runner.
    Copyright 2012, Alex Beal — Used by permission
    """
    g = gen(*args, **kwargs)
    while isinstance(g, types.GeneratorType):
        g = next(g)
    return g


def sum_tramp(lst, acc=0):
    if not lst:
        yield acc
    else:
        yield sum_tramp(lst[1:], acc + lst[0])


# Works for large lists without hitting recursion limit
large_list = list(range(500))
result = tramp(sum_tramp, large_list)
print('Trampolined sum of 0..499:', result)
print('Expected:', sum(large_list))

Trampolined sum of 0..499: 124750
Expected: 124750


In [22]:
# ── functools.reduce: fold a sequence to a single value ───────────────────────
from functools import reduce
import operator

nums = [1, 2, 3, 4, 5]

total = reduce(operator.add, nums)
product = reduce(operator.mul, nums)

print('Sum via reduce:', total)
print('Product via reduce:', product)

# reduce is a higher-order function that models folding — a core FP concept

Sum via reduce: 15
Product via reduce: 120


---

## Module 7 — Eliminating Procedural Code with Pattern Matching

### Pattern Matching in Programming Languages

Pattern matching is a **declarative** way to branch based on the **structure and content** of data, rather than sequences of `if`/`elif` tests.

It exists in many languages:
- **SQL**: `SELECT CASE WHEN ... THEN ...`
- **F#**: `match i with | 42 -> ... | _ -> ...`
- **Haskell**: function definition by cases
- **Python 3.10+**: `match`/`case` statement

### Python `match` Statement Syntax

```python
match <subject>:
    case <pattern1>:
        <action 1>
    case <pattern2> if <guard>:
        <action 2>
    ...
    case _:          # default — matches everything (irrefutable)
        <default action>
```

Supported pattern types:

| Pattern type | Example |
|--------------|---------|
| Literal | `case 42:` |
| Type | `case int():` |
| OR | `case 1 \| 2 \| 3:` |
| Guard | `case x if x > 0:` |
| Capture | `case str(s):` |
| Sequence | `case [1, 2, 3]:` |
| Tuple | `case (x, y):` |
| Mapping | `case {"key": value}:` |
| Class | `case Point(x=0, y=0):` |
| Wildcard | `case _:` |

In [23]:
# ── Basic match on types ──────────────────────────────────────────────────────

def describe(x):
    match x:
        case int():
            return f'{x} is an integer'
        case str():
            return f'{x} is a string'
        case float():
            return f'{x} is a float'
        case Exception():
            return f'An exception: {x}'
        case y:
            return f'Unknown: {y}'

for item in [1, 'hello', 3.14, Exception('oops'), lambda x: x]:
    print(describe(item))

1 is an integer
hello is a string
3.14 is a float
An exception: oops
Unknown: <function <lambda> at 0x10b10ca90>


In [24]:
# ── Match with guards ─────────────────────────────────────────────────────────

for i in range(6):
    match i:
        case 1 | 3 | 5 if i < 4:
            print(f'{i} is odd and less than 4')
        case 1 | 3 | 5 if i >= 4:
            print(f'{i} is odd and >= 4')
        case 2 | 4:
            print(f'{i} is even')
        case 0:
            print(f'{i} is zero')

0 is zero
1 is odd and less than 4
2 is even
3 is odd and less than 4
4 is even
5 is odd and >= 4


In [25]:
# ── Capturing matched values ──────────────────────────────────────────────────

for letter in 'ABC':
    match letter:
        case 'A' as letter_a:
            print(f'{letter!r} captured as {letter_a!r}')
        case str(s) if letter == 'B':
            print(f'{letter!r} captured as str {s!r}')
        case other:
            print(f'Default: {other!r}')

'A' captured as 'A'
'B' captured as str 'B'
Default: 'C'


In [26]:
# ── Structural matching: lists, tuples, mappings ──────────────────────────────

for value in [[1, 2, 3], (4, 5, 6), {'a': 42}]:
    match value:
        case [1, 2, 3] as my_list:
            print(f'{my_list} is a list')
        case (_, 5 as mid, _) as my_tuple:
            print(f'{my_tuple} is a tuple with middle {mid}')
        case {'a': 42 as answer}:
            print(f'The answer is {answer}')

[1, 2, 3] is a list
(4, 5, 6) is a tuple with middle 5
The answer is 42


In [27]:
# ── Class pattern matching with dataclasses ───────────────────────────────────
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

@dataclass
class Circle:
    center: Point
    radius: float

shapes = [Point(0, 0), Point(1, 2), Circle(Point(0, 0), 5.0)]

for shape in shapes:
    match shape:
        case Point(x=0, y=0):
            print('Origin point')
        case Point(x=x, y=y):
            print(f'Point at ({x}, {y})')
        case Circle(center=Point(x=cx, y=cy), radius=r):
            print(f'Circle centred at ({cx},{cy}) radius {r}')

Origin point
Point at (1, 2)
Circle centred at (0,0) radius 5.0


In [28]:
# ── Validation with pattern matching (from course demos) ─────────────────────
# Replace chains of isinstance() + if checks with structured patterns

from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    max_retries: int
    timeout: str
    verbose: bool

    def __post_init__(self):
        match (self.max_retries, self.timeout, self.verbose):
            case (int(r), str(t), bool(v)) if r > 0 and t:
                pass  # valid
            case (int(r), _, _) if r <= 0:
                raise ValueError(f'max_retries must be > 0, got {r}')
            case (_, str(t), _) if not t:
                raise ValueError('timeout must be non-empty')
            case _:
                raise ValueError('Invalid configuration')

# Valid
cfg = Config(max_retries=3, timeout='30s', verbose=True)
print('Valid config:', cfg)

# Invalid
for args in [(0, '30s', True), (3, '', True)]:
    try:
        Config(*args)
    except ValueError as e:
        print('Validation error:', e)

Valid config: Config(max_retries=3, timeout='30s', verbose=True)
Validation error: max_retries must be > 0, got 0
Validation error: timeout must be non-empty


---

## Bringing It All Together

The following example applies **all six FP principles** in a small order-processing pipeline.

In [29]:
from dataclasses import dataclass, replace
from typing import Callable, Iterator
from functools import reduce

# ── Immutable data model (Principle 3) ────────────────────────────────────────
@dataclass(frozen=True)
class OrderItem:
    name: str
    quantity: int
    price: float

@dataclass(frozen=True)
class Order:
    order_id: int
    items: tuple   # tuple keeps it immutable
    status: str = 'pending'


# ── Pure functions (Principle 2) ──────────────────────────────────────────────
def item_total(item: OrderItem) -> float:
    return item.quantity * item.price

def order_total(order: Order) -> float:
    return reduce(lambda acc, item: acc + item_total(item), order.items, 0.0)

def apply_discount(order: Order, rate: float) -> Order:
    discounted = tuple(replace(item, price=item.price * (1 - rate)) for item in order.items)
    return replace(order, items=discounted)

def ship_order(order: Order) -> Order:
    return replace(order, status='shipped')


# ── First-class functions — pipeline (Principle 1) ────────────────────────────
def pipeline(value, *transforms: Callable):
    return reduce(lambda v, fn: fn(v), transforms, value)


# ── Lazy evaluation — generate orders on demand (Principle 4) ────────────────
def order_generator(n: int) -> Iterator[Order]:
    for i in range(1, n + 1):
        items = tuple(
            OrderItem(name=f'item-{j}', quantity=j, price=10.0 * j)
            for j in range(1, 4)
        )
        yield Order(order_id=i, items=items)


# ── Pattern matching — categorise order (Principle 6) ────────────────────────
def categorise_order(order: Order) -> str:
    total = order_total(order)
    match (order.status, total):
        case ('shipped', t) if t > 100:
            return 'VIP shipped'
        case ('shipped', _):
            return 'standard shipped'
        case ('pending', t) if t > 100:
            return 'VIP pending'
        case _:
            return 'standard pending'


# ── Run the pipeline ──────────────────────────────────────────────────────────
from functools import partial

apply_10pct = partial(apply_discount, rate=0.10)

orders = list(order_generator(3))   # materialise the lazy generator

for order in orders:
    processed = pipeline(order, apply_10pct, ship_order)
    total = order_total(processed)
    category = categorise_order(processed)
    print(f'Order {processed.order_id}: ${total:.2f} — {category}')

Order 1: $126.00 — VIP shipped
Order 2: $126.00 — VIP shipped
Order 3: $126.00 — VIP shipped


---

## Summary

| Principle | Python tools |
|-----------|-------------|
| **First-class functions** | `def`, `lambda`, `functools.partial`, `map`, `filter` |
| **Pure functions** | Plain `def` with no side effects; single-purpose functions |
| **Immutability** | `tuple`, `frozenset`, `@dataclass(frozen=True)`, `namedtuple`, `types.MappingProxyType` |
| **Lazy evaluation** | generators, `yield`, `yield from`, `map`/`filter` (Python 3), `itertools` |
| **Recursion** | recursive `def`, tail-call accumulator pattern, trampolining via generators |
| **Pattern matching** | `match`/`case` (Python 3.10+), structural patterns, guards, `@singledispatch` |

### Key Takeaways

- Functional programming **reduces bugs** by eliminating shared mutable state.
- Pure functions are **easy to test** — given inputs, verify outputs.
- Lazy evaluation **defers work** until results are needed, saving memory and time.
- Recursion + trampolining gives you the **expressiveness of recursion** without Python's stack limits.
- Pattern matching replaces **complex if/elif chains** with readable, structural code.
- Python lets you **mix FP and OOP** — you do not have to choose one exclusively.